# Demo 12 — Dal segmento alla colonna

Al primo avvio usare **Run → Run All Cells** (▶▶): la preparazione carica `audiolab`. Poi modificare la cella della demo e rieseguirla con **Shift + Invio**. Ogni riquadro è indipendente.

Le funzioni `al.…` sono quelle della libreria del corso. La prima esecuzione richiede il download del runtime e delle dipendenze.

In [ ]:
# Preparazione automatica: la stessa libreria audiolab dei laboratori.
import sys
from pathlib import Path
if sys.platform == "emscripten":
    import micropip
    await micropip.install(["numpy", "scipy", "soundfile", "matplotlib", "plotly==6.3.0", "nbformat>=5.10"])
    from pyodide.http import pyfetch
    from js import location
    import io, zipfile
    ROOT = Path("/home/pyodide/infomm-3485c94980f2f6df")
    if not (ROOT / "lab-2026/audiolab/__init__.py").exists():
        prefisso = "/infomm-audio-AI" if location.hostname.endswith("github.io") else ""
        url = str(location.origin) + prefisso + "/_jupyterlite/files/audiolab-course.zip?v=3485c94980f2f6df"
        risposta = await pyfetch(url)
        if not risposta.ok:
            raise RuntimeError("Impossibile caricare audiolab: " + str(risposta.status))
        with zipfile.ZipFile(io.BytesIO(await risposta.bytes())) as archivio:
            archivio.extractall(ROOT)
else:
    ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "lab-2026/audiolab").is_dir()), None)
    if ROOT is None:
        raise FileNotFoundError("Avviare Jupyter dalla cartella InfoMM-local del corso.")
sys.path.insert(0, str(ROOT / "lab-2026"))
import audiolab as al
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy import signal
from IPython.display import Audio, Markdown, display
import time
pio.renderers.default = "plotly_mimetype"
AUDIO = al.AUDIO_DIR
DEMO_ASSETS = ROOT / "lab-2026/interattivo/assets"
SR = FS = 22050
N_FFT, HOP = 2048, 512
BLU, ARANCIO, VERDE = "#1766A5", "#D86822", "#277668"
print("audiolab", al.__version__, "— ambiente pronto")


In [ ]:
sr_vis = 16000
L_vis, H_vis = 400, 160
t_vis = np.arange(sr_vis) / sr_vis
y_vis = 0.4 * signal.chirp(t_vis, f0=300, f1=2400, t1=1, method="linear")
w_vis = signal.get_window("hann", L_vis, fftbins=True)
D_vis = al.analysis.stft(y_vis, n_fft=L_vis, win_length=L_vis,
                     hop_length=H_vis, window=w_vis, center=False)
f_vis = np.fft.rfftfreq(L_vis, 1 / sr_vis)
frames_vis = [40, 41]
fig = make_subplots(rows=2, cols=2,
                    subplot_titles=("1. Due segmenti sovrapposti", "2. Segmento × finestra Hann",
                                    "3. FFT di ciascun segmento", "4. Ogni spettro diventa una colonna"))
fig.add_trace(go.Scatter(x=t_vis, y=y_vis, mode="lines", line=dict(width=0.7, color="#898989"),
                         showlegend=False), row=1, col=1)
for m, colore in zip(frames_vis, ["#1976B5", "#D47426"]):
    inizio = m * H_vis
    segmento = y_vis[inizio:inizio + L_vis]
    pesato = segmento * w_vis
    coeff = np.fft.rfft(pesato)
    np.testing.assert_allclose(coeff, D_vis[:, m], atol=1e-12)
    fig.add_vrect(x0=inizio / sr_vis, x1=(inizio + L_vis) / sr_vis, fillcolor=colore,
                  opacity=0.25, line_width=0, row=1, col=1)
    fig.add_trace(go.Scatter(x=np.arange(L_vis) / sr_vis * 1000, y=pesato, mode="lines",
                             line=dict(width=1.2, color=colore), name=f"frame {m}"), row=1, col=2)
    fig.add_trace(go.Scatter(x=f_vis, y=abs(coeff), mode="lines", line=dict(width=1.2, color=colore),
                             name=f"colonna {m}", showlegend=False), row=2, col=1)
db_vis = al.analysis.amplitude_to_db(abs(D_vis), ref=np.max)
fig.add_trace(go.Heatmap(x=np.arange(D_vis.shape[1]) * H_vis / sr_vis, y=f_vis, z=db_vis,
                         colorscale="Magma", zmin=-80, zmax=0, showscale=False), row=2, col=2)
for m, colore in zip(frames_vis, ["#1976B5", "#D47426"]):
    fig.add_vline(x=m * H_vis / sr_vis, line=dict(color=colore, width=2), row=2, col=2)
fig.update_xaxes(range=[0.38, 0.46], title_text="tempo nel segnale [s]", row=1, col=1)
fig.update_xaxes(title_text="tempo nel segmento [ms]", row=1, col=2)
fig.update_xaxes(range=[0, 3000], title_text="frequenza [Hz]", row=2, col=1)
fig.update_xaxes(title_text="inizio frame [s]", row=2, col=2)
fig.update_yaxes(range=[0, 3000], row=2, col=2)
fig.update_layout(height=640, title="Dal segmento alla colonna")
fig.show()
print("Forma della matrice (bin, frame):", D_vis.shape)
print("Durata osservata:", 1000*L_vis/sr_vis, "ms; passo:", 1000*H_vis/sr_vis, "ms")
